# 05. PyTorch Optimizers and Loss - 练习

**难度：** Medium | **标签：** `PyTorch`, `Loss`, `Optimizer` | **目标人群：** PyTorch 入门学习者

本练习配套导学：[Chapter 0 导学](./intro.md)

## 🎯 学习目标
- 掌握 MSE / CrossEntropy 等常见损失
- 理解 SGD / Adam 的基本使用
- 让一个简单模型完成一步训练更新


## Part 1: 损失函数

### 练习 1.1: 手写 MSE Loss
实现均方误差损失，并用于一个小回归样例。


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F


def mse_loss(pred, target):
    return torch.mean((pred - target) ** 2)


def cross_entropy_loss(logits, target):
    return F.cross_entropy(logits, target)


def train_one_step(model, x, target, optimizer):
    model.train()
    optimizer.zero_grad()
    pred = model(x)
    loss = mse_loss(pred, target)
    loss.backward()
    optimizer.step()
    return loss.item()


In [ ]:
def test_mse_loss():
    pred = torch.tensor([[1.0, 3.0], [2.0, 4.0]])
    target = torch.tensor([[1.0, 1.0], [3.0, 5.0]])
    loss = mse_loss(pred, target)
    assert torch.allclose(loss, torch.tensor(1.5))
    print('✅ mse_loss 通过')


def test_cross_entropy_loss():
    logits = torch.tensor([[2.0, 0.5], [0.1, 1.2]])
    target = torch.tensor([0, 1])
    loss = cross_entropy_loss(logits, target)
    expected = F.cross_entropy(logits, target)
    assert torch.allclose(loss, expected)
    print('✅ cross_entropy_loss 通过')


def test_train_one_step():
    model = nn.Linear(1, 1)
    with torch.no_grad():
        model.weight.zero_()
        model.bias.zero_()
    x = torch.tensor([[1.0], [2.0], [3.0]])
    target = 2 * x + 1
    optimizer = torch.optim.SGD(model.parameters(), lr=0.1)
    before = mse_loss(model(x), target).item()
    loss = train_one_step(model, x, target, optimizer)
    after = mse_loss(model(x), target).item()
    assert abs(loss - before) < 1e-9
    assert after < before
    print('✅ train_one_step 通过')


test_mse_loss()
test_cross_entropy_loss()
test_train_one_step()


## Part 2: 实战应用

### 练习 2.1: 一步训练演示
观察 SGD 更新前后损失是否下降。


In [ ]:
model = nn.Linear(1, 1)
with torch.no_grad():
    model.weight.zero_()
    model.bias.zero_()

x = torch.tensor([[1.0], [2.0], [3.0], [4.0]])
target = 2 * x + 1
optimizer = torch.optim.SGD(model.parameters(), lr=0.05)

print('初始损失:', mse_loss(model(x), target).item())
for step in range(3):
    loss = train_one_step(model, x, target, optimizer)
    print(f'step {step + 1}: loss={loss:.4f}')
print('训练后预测:', model(x).detach().squeeze().tolist())
